# Modeling — Random Forest (quantile 0.9)

Desain: `docs/superpowers/specs/2026-08-18-random-forest-modeling-design.md`.
Rencana: `docs/superpowers/plans/2026-08-18-random-forest-modeling.md`.

Notebook ini tipis dengan sengaja. Semua logika ada di `utils/walk_forward.py`
dan `utils/model_random_forest.py`, supaya jalur skrip dan jalur notebook tidak
bisa berbeda — persis kesalahan yang pernah terjadi di `data-processing.ipynb`.

**Desember 2025 terkunci** dan tidak dinilai di sini.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from utils.modelling import evaluation, model_random_forest as rf
from utils.modelling import modeling_prep, run_config, walk_forward

# Random Forest tidak punya jalur GPU — `quantile-forest` murni CPU, jadi
# tidak ada FORECAST_DEVICE yang berarti di sini dan modelnya dijalankan
# di Mac lokal (Bagian 2 spec eksekusi terdistribusi). Setelan environment
# yang lain tetap dibaca supaya ketiga notebook berperilaku sama, dan
# supaya `device` tetap tercatat di tiap baris hasil sebagai "cpu" alih-alih
# kosong. Tanpa satu pun env var, perilakunya persis seperti sebelumnya.
DEVICE = run_config.device("cpu")
SHARD = run_config.shard()
SEARCH_FILE = run_config.search_checkpoint("rf")
RESULTS_FILE = run_config.checkpoint_path("rf_walk_forward_results.csv")

df = pd.read_parquet(run_config.model_input_path())
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(run_config.describe(DEVICE))


## Benchmark

Satu fit di training set penuh fold 5, untuk memastikan batas leaf storage
berlaku dan menentukan ukuran pencarian. Angkanya dicatat di
`docs/hasil-modeling-rf.md`.

In [ ]:
import resource
import time

split = walk_forward.prepare_fold(df, 5)
train, valid = split["train"], split["valid"]
print(f"train {len(train):,} rows, valid {len(valid):,} rows")
print(f"QUANTILE_SET: {len(rf.QUANTILES)} titik, {rf.QUANTILES[0]}..{rf.QUANTILES[-1]}")

params = dict(rf.DEFAULT_PARAMS)
print("estimated leaf storage: "
      f"{rf.estimate_leaf_memory_bytes(params, len(train)) / 1024 ** 3:.2f} GB")

start = time.time()
prediction = rf.make_fit_predict(params)(train, valid)
elapsed = time.time() - start

peak_bytes = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss  # bytes on macOS
headline = min(range(len(rf.QUANTILES)),
               key=lambda i: abs(rf.QUANTILES[i] - evaluation.DEFAULT_ALPHA))
print(f"wall time {elapsed / 60:.1f} min")
print(f"peak RSS  {peak_bytes / 1024 ** 3:.2f} GB")
print(f"prediction shape {prediction.shape} (baris x titik kuantil)")
print(f"di tau=0.9: mean {prediction[:, headline].mean():.2f}, "
      f"max {prediction[:, headline].max():.2f}")
# Nol secara struktural: setiap titik adalah persentil dari satu distribusi
# daun yang sama. Dicetak sebagai bukti, bukan sebagai harapan.
print(f"crossing_rate {evaluation.crossing_rate(prediction, rf.QUANTILES):.4f}")


## Pencarian hyperparameter

18 kandidat, tersaring budget memori, dinilai di fold 3 dan 5 dengan pinball@0.9
gabungan. Hasil yang dilaporkan datang dari walk-forward lima fold di bawah,
bukan dari sini — menilai di fold yang memilih pemenang akan optimistis.

In [ ]:
# Butir 0c: pencarian hyperparameter RF **dijalankan ulang** (keputusan
# pemilik proyek 2026-08-24, membalik revisi sebelumnya).
#
# Alasan revisi lama masih benar dan tidak dibantah: hyperparameter forest
# membentuk *daun*, dan seluruh titik QUANTILE_SET dibaca dari daun yang sama,
# jadi migrasi multi-kuantil sendiri tidak mengubah pertanyaan yang dijawab
# pencarian ini. Yang membalikkannya adalah datanya, bukan kriterianya:
# rf_best_params.json yang lama dipilih 2026-08-18, sebelum reclass WIP-2 masuk
# ke artefak (dibangun ulang 2026-08-23 22:52) — kebasian yang sama yang sudah
# dipakai sebagai alasan membuang bundle terlatihnya.
# Lihat 2026-08-18-random-forest-modeling-design.md Part 2.
#
# Anggaran tidak berubah: 18 kandidat, SEARCH_FOLDS = (3, 5), seed 42.
#
# Prasyarat langkah 0 Fase 3 sudah dipenuhi: rf_search_results.csv DAN
# rf_best_params.json dari run kuantil-tunggal diganti nama menjadi
# `*.single-quantile.bak.*` (2026-08-24) sesudah guard checkpoint diverifikasi
# berbunyi — RF berhenti setelah 2,8 detik. Kalau berkas tanpa kolom
# `headline_quantile` muncul lagi di jalur checkpoint, guard yang sama di
# model_common._assert_checkpoint_matches() akan menolaknya lagi.
train_size = len(walk_forward.prepare_fold(df, 5)["train"])
candidates = rf.sample_search_space(18, n_train=train_size, seed=42)
search_results = rf.run_search(df, candidates, folds=rf.SEARCH_FOLDS,
                               checkpoint_path=SEARCH_FILE, only=SHARD,
                               provenance=run_config.provenance(DEVICE))
search_results.to_csv(SEARCH_FILE, index=False)

# Pemenang dipilih di sel ini, jadi guard shard-nya juga di sini: memilih
# dari sebagian kandidat menghasilkan angka yang tampak sepenuhnya wajar.
assert SHARD is None, (
    "run bershard: jangan pilih pemenang dari sebagian kandidat — "
    "gabungkan seluruh shard dengan model_common.merge_shards() lebih dulu"
)
best = rf.select_best(search_results, candidates)

print(best)


## Walk-forward final

Konfigurasi pemenang di kelima fold, melawan ketiga baseline naive pada baris
yang identik.

In [ ]:
rf.save_best_params(best)

fit_predict = rf.make_fit_predict(best)
results = walk_forward.run_walk_forward(df, fit_predict, model_name="random_forest",
                                        quantiles=rf.QUANTILES)
results.to_csv(RESULTS_FILE, index=False)

print(f"K1 (rata-rata pinball lintas {len(rf.QUANTILES)} kuantil): "
      f"{walk_forward.pooled_k1(results, 'random_forest'):.4f}")

overall = results[results["group_col"].isna()]
(overall[(overall["quantile"] - evaluation.DEFAULT_ALPHA).abs() < 1e-9]
 .pivot_table(index="model", columns="fold_id", values="pinball").round(3))


In [ ]:
bundle = rf.fit_final(df, best)
rf.save_bundle(bundle)
print(f"trained on {bundle['n_train']:,} rows, "
      f"{len(bundle['columns'])} columns, "
      f"{len(bundle['quantiles'])} titik kuantil "
      f"({bundle['quantiles'][0]}..{bundle['quantiles'][-1]})")


## Hasil

Tiga potongan, masing-masing melawan ketiga baseline naive pada baris identik.
Satu angka global menyesatkan di data yang 45% targetnya nol.

In [ ]:
results = pd.read_csv(rf.RESULTS_FILE)
HEADLINE = evaluation.DEFAULT_ALPHA

print("=== K1 (rata-rata pinball lintas QUANTILE_SET, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} K1 {walk_forward.pooled_k1(results, model):7.4f}")

print("\n=== per fold, K1 ===")
print(results[results["group_col"].isna()]
      .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

print(f"\n=== per fold, pinball di tau={HEADLINE} (angka headline B-9) ===")
headline_rows = results[results["group_col"].isna()
                        & ((results["quantile"] - HEADLINE).abs() < 1e-9)]
print(headline_rows.pivot_table(index="model", columns="fold_id",
                                values="pinball").round(3))

for group_col in walk_forward.GROUP_COLS:
    print(f"\n=== per {group_col} (K1, pooled over folds) ===")
    grouped = results[results["group_col"] == group_col]
    # Kolom dipilih sebelum apply: tanpa itu pandas ikut menyertakan kolom
    # pengelompokan dan mengeluarkan FutureWarning di setiap sel.
    table = (grouped.assign(weighted=grouped["pinball"] * grouped["n"])
                    .groupby(["model", "group_value"], observed=True)[["weighted", "n"]]
                    .apply(lambda part: part["weighted"].sum() / part["n"].sum())
                    .unstack())
    print(table.round(3))

# pooled_metric menolak dirata-ratakan lintas kuantil untuk metrik selain
# pinball/crossing_rate — coverage di 0,05 dan di 0,95 menjawab pertanyaan
# yang berbeda. Jadi ketiganya dibaca di tau headline, eksplisit.
print(f"\n=== coverage / fill rate di tau={HEADLINE} (overall, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} "
          f"coverage {walk_forward.pooled_metric(results, model, 'coverage', quantile=HEADLINE):6.3f}  "
          f"fill_rate {walk_forward.pooled_metric(results, model, 'fill_rate', quantile=HEADLINE):6.3f}  "
          f"shortfall {walk_forward.pooled_metric(results, model, 'shortfall_units', quantile=HEADLINE):9.1f}  "
          f"crossing {walk_forward.pooled_metric(results, model, 'crossing_rate'):6.4f}")

print("\n=== K2: coverage per titik kuantil (random_forest) ===")
print(walk_forward.coverage_by_quantile(results, "random_forest").round(4)
      .to_string(index=False))
